In [5]:
import os
import tkinter as tk
from tkinter import ttk
from tkinter import messagebox as msgbox

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageTk
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib.figure import Figure

# =========================
# Matplotlib 中文字型
# =========================
plt.rcParams["font.family"] = ["Microsoft JhengHei", "PingFang TC", "Arial Unicode MS", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False
img_path = r"D:\桌面\一番賞.png"
# =========================
# 全域資料
# =========================
result = []
vas = []
much = []

count = {}
total_tickets = 0
price_1 = 0
price_2 = 0
price_3 = 0
p1 = 0
p2 = 0
p3 = 0

# 票數設定視窗會建立的元件
A = B = C = D = E = F = G = None
m1 = None
c1 = c2 = c3 = None
combobox = None
combo1 = combo2 = combo3 = None
pc = ["A賞", "B賞", "C賞", "D賞", "E賞", "F賞", "G賞"]
tree1 = None

# =========================
# UI 色彩
# =========================
BG = "#F5F7FB"
CARD = "#FFFFFF"
PRIMARY = "#6C63FF"
PRIMARY_DARK = "#574FD6"
ACCENT = "#FF8A65"
TEXT = "#1F2937"
MUTED = "#6B7280"
BORDER = "#E5E7EB"
SUCCESS = "#10B981"
DANGER = "#EF4444"


def center_window(window, width, height):
    """讓視窗顯示在螢幕中央。"""
    window.update_idletasks()
    sw = window.winfo_screenwidth()
    sh = window.winfo_screenheight()
    x = max((sw - width) // 2, 0)
    y = max((sh - height) // 2, 0)
    window.geometry(f"{width}x{height}+{x}+{y}")


def setup_styles():
    style = ttk.Style()
    try:
        style.theme_use("clam")
    except tk.TclError:
        pass

    style.configure(
        "TButton",
        font=("Microsoft JhengHei UI", 11),
        padding=(14, 10),
        borderwidth=0,
    )

    style.configure(
        "Primary.TButton",
        foreground="white",
        background=PRIMARY,
        font=("Microsoft JhengHei UI", 11, "bold"),
        padding=(18, 12),
        borderwidth=0,
    )
    style.map(
        "Primary.TButton",
        background=[("active", PRIMARY_DARK), ("pressed", PRIMARY_DARK)],
    )

    style.configure(
        "Secondary.TButton",
        foreground=TEXT,
        background="#EEF0FF",
        font=("Microsoft JhengHei UI", 11, "bold"),
        padding=(18, 12),
        borderwidth=0,
    )
    style.map("Secondary.TButton", background=[("active", "#E2E5FF")])

    style.configure(
        "Danger.TButton",
        foreground="white",
        background=DANGER,
        font=("Microsoft JhengHei UI", 11, "bold"),
        padding=(18, 12),
        borderwidth=0,
    )
    style.map("Danger.TButton", background=[("active", "#DC2626")])

    style.configure(
        "TEntry",
        padding=7,
        fieldbackground="white",
        bordercolor=BORDER,
        lightcolor=BORDER,
        darkcolor=BORDER,
    )

    style.configure(
        "TCombobox",
        padding=7,
        fieldbackground="white",
        background="white",
        bordercolor=BORDER,
        arrowsize=15,
    )

    style.configure(
        "Treeview",
        rowheight=32,
        font=("Microsoft JhengHei UI", 10),
        background="white",
        fieldbackground="white",
        foreground=TEXT,
        borderwidth=0,
    )
    style.configure(
        "Treeview.Heading",
        font=("Microsoft JhengHei UI", 10, "bold"),
        background="#EEF0FF",
        foreground=TEXT,
        relief="flat",
        padding=8,
    )
    style.map("Treeview.Heading", background=[("active", "#E2E5FF")])


def make_card(parent, **kwargs):
    return tk.Frame(
        parent,
        bg=CARD,
        highlightbackground=BORDER,
        highlightthickness=1,
        bd=0,
        **kwargs,
    )


def section_title(parent, title, subtitle=None):
    box = tk.Frame(parent, bg=CARD)
    tk.Label(
        box,
        text=title,
        font=("Microsoft JhengHei UI", 15, "bold"),
        bg=CARD,
        fg=TEXT,
    ).pack(anchor="w")
    if subtitle:
        tk.Label(
            box,
            text=subtitle,
            font=("Microsoft JhengHei UI", 9),
            bg=CARD,
            fg=MUTED,
        ).pack(anchor="w", pady=(3, 0))
    return box


# =========================
# 數據表
# =========================
def g_data():
    global tree1

    if tree1 is None or not tree1.winfo_exists():
        return

    if len(result) == 0 or len(vas) == 0 or len(much) == 0:
        msgbox.showerror("錯誤", "沒有可顯示的數據，請先執行視覺化分析！")
        return

    for item in tree1.get_children():
        tree1.delete(item)

    for i, (price, tickets, expectation) in enumerate(zip(result, vas, much)):
        tree1.insert("", "end", iid=str(i), values=(price, tickets, f"{expectation:.2f}"))

    tree1.insert(
        "",
        "end",
        values=("總和", f"{sum(vas):,}", f"{sum(much):.2f}"),
        tags=("total",),
    )
    tree1.tag_configure("total", background="#F3F4FF", font=("Microsoft JhengHei UI", 10, "bold"))


def g_simu():
    global tree1

    w_5 = tk.Toplevel(win)
    w_5.title("一番賞模擬與期望值數據")
    w_5.configure(bg=BG)
    center_window(w_5, 820, 600)
    w_5.minsize(700, 450)

    container = tk.Frame(w_5, bg=BG)
    container.pack(fill="both", expand=True, padx=24, pady=24)

    header = tk.Frame(container, bg=BG)
    header.pack(fill="x", pady=(0, 16))

    tk.Label(
        header,
        text="期望值資料表",
        font=("Microsoft JhengHei UI", 22, "bold"),
        bg=BG,
        fg=TEXT,
    ).pack(side="left")

    ttk.Button(
        header,
        text="重新載入資料",
        style="Primary.TButton",
        command=g_data,
    ).pack(side="right")

    tk.Label(
        container,
        text="先執行 3D 預測視覺化，再回到此處查看所有價格、購買張數與理論期望值。",
        font=("Microsoft JhengHei UI", 10),
        bg=BG,
        fg=MUTED,
    ).pack(anchor="w", pady=(0, 12))

    card = make_card(container)
    card.pack(fill="both", expand=True)

    table_wrap = tk.Frame(card, bg=CARD)
    table_wrap.pack(fill="both", expand=True, padx=18, pady=18)
    table_wrap.rowconfigure(0, weight=1)
    table_wrap.columnconfigure(0, weight=1)

    columns = ("價格", "票數", "購買理論期望值")
    tree1 = ttk.Treeview(table_wrap, show="headings", columns=columns)

    tree1.heading("價格", text="單張價格")
    tree1.heading("票數", text="購買張數")
    tree1.heading("購買理論期望值", text="理論期望值")

    tree1.column("價格", anchor="center", width=160, minwidth=120)
    tree1.column("票數", anchor="center", width=160, minwidth=120)
    tree1.column("購買理論期望值", anchor="center", width=260, minwidth=180)

    y_scroll = ttk.Scrollbar(table_wrap, orient="vertical", command=tree1.yview)
    x_scroll = ttk.Scrollbar(table_wrap, orient="horizontal", command=tree1.xview)
    tree1.configure(yscrollcommand=y_scroll.set, xscrollcommand=x_scroll.set)

    tree1.grid(row=0, column=0, sticky="nsew")
    y_scroll.grid(row=0, column=1, sticky="ns")
    x_scroll.grid(row=1, column=0, sticky="ew")

    if result:
        g_data()


# =========================
# 3D 視覺化
# =========================
def d_sim():
    global result, vas, much

    if not count or total_tickets == 0:
        msgbox.showerror("錯誤", "請先進行票數設定並按下『計算期望值』！")
        return

    try:
        combo1_value = combo1.get()
        combo2_value = combo2.get()
        combo3_value = combo3.get()

        c1_val = c1.get()
        c2_val = c2.get()
        c3_val = c3.get()
    except Exception:
        msgbox.showerror("錯誤", "請先完成票數設定！")
        return

    like = list(range(100, 501, 50))
    ticket_range = list(range(1, 82, 10))

    result = []
    vas = []
    much = []

    for price in like:
        for tickets in ticket_range:
            for val, combo in zip(
                [c1_val, c2_val, c3_val],
                [combo1_value, combo2_value, combo3_value],
            ):
                expect = val * count[combo] / total_tickets * tickets
                cost = price * tickets
                net = round(expect - cost, 2)

                result.append(price)
                vas.append(tickets)
                much.append(net)

    w_3 = tk.Toplevel(win)
    w_3.title("3D 視覺化分析")
    w_3.configure(bg=BG)
    center_window(w_3, 980, 760)
    w_3.minsize(760, 600)

    header = tk.Frame(w_3, bg=BG)
    header.pack(fill="x", padx=24, pady=(20, 8))

    tk.Label(
        header,
        text="3D 期望值視覺化",
        font=("Microsoft JhengHei UI", 20, "bold"),
        bg=BG,
        fg=TEXT,
    ).pack(anchor="w")
    tk.Label(
        header,
        text="觀察單張價格、購買張數與期望值之間的變化。",
        font=("Microsoft JhengHei UI", 10),
        bg=BG,
        fg=MUTED,
    ).pack(anchor="w", pady=(3, 0))

    card = make_card(w_3)
    card.pack(fill="both", expand=True, padx=24, pady=(8, 24))

    fig = Figure(figsize=(9, 6.5), dpi=100, facecolor="white")
    ax = fig.add_subplot(111, projection="3d")

    values_for_color = np.array(much)
    scatter = ax.scatter(
        result,
        vas,
        much,
        c=values_for_color,
        cmap="viridis",
        alpha=0.82,
        s=30,
    )

    ax.set_xlabel("單張價格", fontsize=11, labelpad=10)
    ax.set_ylabel("購買張數", fontsize=11, labelpad=10)
    ax.set_zlabel("期望值", fontsize=11, labelpad=10)
    ax.set_title("一番賞預測期望值 3D 分布", fontsize=14, pad=18)
    ax.grid(True, alpha=0.25)
    fig.colorbar(scatter, ax=ax, shrink=0.6, pad=0.1, label="期望值")
    fig.tight_layout()

    canvas = FigureCanvasTkAgg(fig, master=card)
    canvas.draw()
    canvas.get_tk_widget().pack(fill="both", expand=True, padx=10, pady=10)


# =========================
# 長條圖
# =========================
def s_n1():
    try:
        ticket = [A.get(), B.get(), C.get(), D.get(), E.get(), F.get(), G.get()]
    except Exception:
        msgbox.showerror("錯誤", "請先開啟票數設定！")
        return

    plot_win = tk.Toplevel(win)
    plot_win.title("獎品數量長條圖")
    plot_win.configure(bg=BG)
    center_window(plot_win, 760, 560)

    tk.Label(
        plot_win,
        text="獎品剩餘數量",
        font=("Microsoft JhengHei UI", 20, "bold"),
        bg=BG,
        fg=TEXT,
    ).pack(anchor="w", padx=24, pady=(20, 4))
    tk.Label(
        plot_win,
        text="快速比較 A～G 賞目前的剩餘張數。",
        font=("Microsoft JhengHei UI", 10),
        bg=BG,
        fg=MUTED,
    ).pack(anchor="w", padx=24, pady=(0, 12))

    card = make_card(plot_win)
    card.pack(fill="both", expand=True, padx=24, pady=(0, 24))

    fig = Figure(figsize=(7, 4.5), dpi=100, facecolor="white")
    ax = fig.add_subplot(111)
    bars = ax.bar(pc, ticket)
    ax.set_title("獎品數量分佈", fontsize=14, pad=14)
    ax.set_xlabel("獎品種類")
    ax.set_ylabel("張數")
    ax.grid(axis="y", alpha=0.2)

    for bar, value in zip(bars, ticket):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            str(value),
            ha="center",
            va="bottom",
            fontsize=9,
        )

    fig.tight_layout()
    canvas = FigureCanvasTkAgg(fig, master=card)
    canvas.draw()
    canvas.get_tk_widget().pack(fill="both", expand=True, padx=10, pady=10)


# =========================
# 期望值計算
# =========================
def s_n():
    global count, p1, p2, p3, total_tickets, price_1, price_2, price_3

    try:
        count = {
            "A賞": A.get(),
            "B賞": B.get(),
            "C賞": C.get(),
            "D賞": D.get(),
            "E賞": E.get(),
            "F賞": F.get(),
            "G賞": G.get(),
        }

        total_tickets = sum(count.values())
        if total_tickets <= 0:
            msgbox.showerror("錯誤", "總票數必須大於 0！")
            return

        selected = [combo1.get(), combo2.get(), combo3.get()]
        if any(item not in pc for item in selected):
            msgbox.showerror("錯誤", "請正確選擇前三名想要的獎品！")
            return

        # 保留原本的邏輯：三個偏好獎品各自搭配右側輸入的市場價格
        price_1 = c1.get()
        price_2 = c2.get()
        price_3 = c3.get()

        buy_count = int(combobox.get())
        price_per_ticket = m1.get()

        prospect = (
            price_1 * count[selected[0]] / total_tickets
            + price_2 * count[selected[1]] / total_tickets
            + price_3 * count[selected[2]] / total_tickets
        ) * buy_count

        p1 = price_1 * count[selected[0]] / total_tickets
        p2 = price_2 * count[selected[1]] / total_tickets
        p3 = price_3 * count[selected[2]] / total_tickets

        cost = buy_count * price_per_ticket
        final = prospect - cost

        if final >= 0:
            result_text = f"預估淨期望值：+{final:.2f} 元"
            detail = "依目前設定，理論期望值高於購買成本。"
        else:
            result_text = f"預估淨期望值：{final:.2f} 元"
            detail = "依目前設定，理論期望值低於購買成本。"

        msgbox.showinfo(
            "期望值計算結果",
            f"{result_text}\n\n"
            f"預估獎品價值：{prospect:.2f} 元\n"
            f"購買成本：{cost:.2f} 元\n\n"
            f"{detail}",
        )

    except (ValueError, tk.TclError):
        msgbox.showerror("輸入錯誤", "請確認所有張數、價格都輸入有效數字。")


# =========================
# 票數設定視窗
# =========================
def dy_simu():
    global A, B, C, D, E, F, G
    global combobox, m1, combo1, combo2, combo3, c1, c2, c3

    w_1 = tk.Toplevel(win)
    w_1.title("一番賞票數與偏好設定")
    w_1.configure(bg=BG)
    center_window(w_1, 900, 660)
    w_1.minsize(820, 620)

    A = tk.IntVar(value=0)
    B = tk.IntVar(value=0)
    C = tk.IntVar(value=0)
    D = tk.IntVar(value=0)
    E = tk.IntVar(value=0)
    F = tk.IntVar(value=0)
    G = tk.IntVar(value=0)
    m1 = tk.IntVar(value=0)
    c1 = tk.IntVar(value=0)
    c2 = tk.IntVar(value=0)
    c3 = tk.IntVar(value=0)

    root = tk.Frame(w_1, bg=BG)
    root.pack(fill="both", expand=True, padx=26, pady=24)

    # 頁首
    tk.Label(
        root,
        text="一番賞期望值設定",
        font=("Microsoft JhengHei UI", 24, "bold"),
        bg=BG,
        fg=TEXT,
    ).pack(anchor="w")
    tk.Label(
        root,
        text="輸入剩餘獎品數量、購買條件，以及你最想要的三個獎品。",
        font=("Microsoft JhengHei UI", 10),
        bg=BG,
        fg=MUTED,
    ).pack(anchor="w", pady=(4, 18))

    # 獎品剩餘張數
    prize_card = make_card(root)
    prize_card.pack(fill="x", pady=(0, 14))

    section_title(
        prize_card,
        "① 剩餘獎品張數",
        "請輸入目前 A～G 賞各自剩餘多少張。",
    ).pack(fill="x", padx=20, pady=(18, 12))

    prize_grid = tk.Frame(prize_card, bg=CARD)
    prize_grid.pack(fill="x", padx=20, pady=(0, 20))

    prize_vars = [("A賞", A), ("B賞", B), ("C賞", C), ("D賞", D), ("E賞", E), ("F賞", F), ("G賞", G)]
    for idx, (name, var) in enumerate(prize_vars):
        cell = tk.Frame(prize_grid, bg=CARD)
        cell.grid(row=0, column=idx, padx=6, sticky="ew")
        prize_grid.columnconfigure(idx, weight=1)

        tk.Label(
            cell,
            text=name,
            font=("Microsoft JhengHei UI", 10, "bold"),
            bg=CARD,
            fg=TEXT,
        ).pack(anchor="w", pady=(0, 5))
        ttk.Entry(cell, textvariable=var, width=8, justify="center").pack(fill="x")

    # 購買條件 + 偏好
    middle = tk.Frame(root, bg=BG)
    middle.pack(fill="both", expand=True, pady=(0, 14))
    middle.columnconfigure(0, weight=1)
    middle.columnconfigure(1, weight=2)
    middle.rowconfigure(0, weight=1)

    buy_card = make_card(middle)
    buy_card.grid(row=0, column=0, sticky="nsew", padx=(0, 7))

    section_title(
        buy_card,
        "② 購買條件",
        "設定你打算抽幾張，以及每張價格。",
    ).pack(fill="x", padx=20, pady=(18, 14))

    buy_form = tk.Frame(buy_card, bg=CARD)
    buy_form.pack(fill="x", padx=20, pady=(0, 20))
    buy_form.columnconfigure(1, weight=1)

    tk.Label(buy_form, text="預算購買張數", font=("Microsoft JhengHei UI", 10), bg=CARD, fg=TEXT).grid(row=0, column=0, sticky="w", pady=8)
    values = list(range(1, 81))
    combobox = ttk.Combobox(buy_form, values=values, state="readonly", width=10)
    combobox.grid(row=0, column=1, sticky="ew", padx=(12, 0), pady=8)
    combobox.set(1)

    tk.Label(buy_form, text="一張的價格", font=("Microsoft JhengHei UI", 10), bg=CARD, fg=TEXT).grid(row=1, column=0, sticky="w", pady=8)
    ttk.Entry(buy_form, textvariable=m1, justify="right").grid(row=1, column=1, sticky="ew", padx=(12, 0), pady=8)

    tk.Label(
        buy_form,
        text="例如：每抽 300 元，打算抽 10 張。",
        font=("Microsoft JhengHei UI", 9),
        bg=CARD,
        fg=MUTED,
    ).grid(row=2, column=0, columnspan=2, sticky="w", pady=(10, 0))

    prefer_card = make_card(middle)
    prefer_card.grid(row=0, column=1, sticky="nsew", padx=(7, 0))

    section_title(
        prefer_card,
        "③ 最想要的獎品",
        "依序選前三名，並輸入各自目前的市場價格。",
    ).pack(fill="x", padx=20, pady=(18, 12))

    pref_form = tk.Frame(prefer_card, bg=CARD)
    pref_form.pack(fill="x", padx=20, pady=(0, 20))
    pref_form.columnconfigure(1, weight=1)
    pref_form.columnconfigure(2, weight=1)

    tk.Label(pref_form, text="順位", font=("Microsoft JhengHei UI", 9, "bold"), bg=CARD, fg=MUTED).grid(row=0, column=0, sticky="w", pady=(0, 5))
    tk.Label(pref_form, text="獎品", font=("Microsoft JhengHei UI", 9, "bold"), bg=CARD, fg=MUTED).grid(row=0, column=1, sticky="w", padx=(10, 0), pady=(0, 5))
    tk.Label(pref_form, text="市場價格", font=("Microsoft JhengHei UI", 9, "bold"), bg=CARD, fg=MUTED).grid(row=0, column=2, sticky="w", padx=(10, 0), pady=(0, 5))

    combo1 = ttk.Combobox(pref_form, values=pc, state="readonly")
    combo2 = ttk.Combobox(pref_form, values=pc, state="readonly")
    combo3 = ttk.Combobox(pref_form, values=pc, state="readonly")
    combo1.set("A賞")
    combo2.set("B賞")
    combo3.set("C賞")

    rows = [
        ("第一名", combo1, c1),
        ("第二名", combo2, c2),
        ("第三名", combo3, c3),
    ]

    for row_idx, (rank, combo, price_var) in enumerate(rows, start=1):
        tk.Label(
            pref_form,
            text=rank,
            font=("Microsoft JhengHei UI", 10, "bold"),
            bg=CARD,
            fg=TEXT,
        ).grid(row=row_idx, column=0, sticky="w", pady=7)
        combo.grid(row=row_idx, column=1, sticky="ew", padx=(10, 6), pady=7)
        ttk.Entry(pref_form, textvariable=price_var, justify="right").grid(row=row_idx, column=2, sticky="ew", padx=(6, 0), pady=7)

    # 底部操作列
    action_bar = tk.Frame(root, bg=BG)
    action_bar.pack(fill="x")

    ttk.Button(
        action_bar,
        text="查看獎品長條圖",
        style="Secondary.TButton",
        command=s_n1,
    ).pack(side="left")

    ttk.Button(
        action_bar,
        text="計算期望值",
        style="Primary.TButton",
        command=s_n,
    ).pack(side="right")


# =========================
# 主畫面圖片
# =========================
def load_home_image(parent):
    desktop_path = os.path.join(os.path.expanduser("~"), "Desktop")
    img_path = r"D:\桌面\一番賞.png"

    holder = tk.Frame(parent, bg="#F0F1FF", width=430, height=270)
    holder.pack_propagate(False)

    if os.path.exists(img_path):
        try:
            img = Image.open(img_path).convert("RGB")
            img.thumbnail((410, 250), Image.Resampling.LANCZOS)
            tk_img = ImageTk.PhotoImage(img, master=win)

            label = tk.Label(holder, image=tk_img, bg="#F0F1FF")
            label.image = tk_img
            label.pack(expand=True)
            return holder, tk_img
        except Exception:
            pass

    tk.Label(
        holder,
        text="🎁\n\n一番賞圖片\n\n將「一番賞.png」放到桌面後\n會自動顯示在這裡",
        font=("Microsoft JhengHei UI", 13),
        bg="#F0F1FF",
        fg=MUTED,
        justify="center",
    ).pack(expand=True)
    return holder, None


# =========================
# 主畫面
# =========================
def build_home():
    container = tk.Frame(win, bg=BG)
    container.pack(fill="both", expand=True, padx=32, pady=28)

    # Header
    top = tk.Frame(container, bg=BG)
    top.pack(fill="x", pady=(0, 20))

    title_box = tk.Frame(top, bg=BG)
    title_box.pack(side="left")

    tk.Label(
        title_box,
        text="一番賞期望值系統",
        font=("Microsoft JhengHei UI", 27, "bold"),
        bg=BG,
        fg=TEXT,
    ).pack(anchor="w")
    tk.Label(
        title_box,
        text="用剩餘票數、市場價格與購買預算，快速估算抽賞的理論期望值。",
        font=("Microsoft JhengHei UI", 10),
        bg=BG,
        fg=MUTED,
    ).pack(anchor="w", pady=(4, 0))

    tk.Label(
        top,
        text="ICHIBAN KUJI  •  EXPECTED VALUE",
        font=("Segoe UI", 9, "bold"),
        bg="#EEF0FF",
        fg=PRIMARY,
        padx=12,
        pady=7,
    ).pack(side="right", anchor="n")

    # Hero
    hero = make_card(container)
    hero.pack(fill="x", pady=(0, 18))

    left = tk.Frame(hero, bg=CARD)
    left.pack(side="left", fill="both", expand=True, padx=(28, 18), pady=28)

    tk.Label(
        left,
        text="抽之前，先算算看。",
        font=("Microsoft JhengHei UI", 25, "bold"),
        bg=CARD,
        fg=TEXT,
    ).pack(anchor="w")

    tk.Label(
        left,
        text="先設定剩餘獎品與你的前三志願，再計算單次購買的理論期望值，最後可以用圖表觀察不同價格與抽數的變化。",
        font=("Microsoft JhengHei UI", 11),
        bg=CARD,
        fg=MUTED,
        justify="left",
        wraplength=430,
        pady=12,
    ).pack(anchor="w")

    buttons = tk.Frame(left, bg=CARD)
    buttons.pack(anchor="w", pady=(8, 0))

    ttk.Button(
        buttons,
        text="開始設定票數",
        style="Primary.TButton",
        command=dy_simu,
    ).pack(side="left", padx=(0, 10))

    ttk.Button(
        buttons,
        text="查看 3D 分析",
        style="Secondary.TButton",
        command=d_sim,
    ).pack(side="left")

    image_holder, home_img = load_home_image(hero)
    image_holder.pack(side="right", padx=(18, 28), pady=28)
    hero.home_img = home_img

    # 功能卡
    cards = tk.Frame(container, bg=BG)
    cards.pack(fill="both", expand=True)
    for i in range(3):
        cards.columnconfigure(i, weight=1, uniform="feature")

    feature_data = [
        (
            "01",
            "票數與偏好設定",
            "輸入 A～G 賞剩餘數量、單抽價格、預算張數與前三志願。",
            "開啟設定",
            dy_simu,
        ),
        (
            "02",
            "3D 期望值分析",
            "比較不同單張價格與購買張數時，淨期望值會如何變化。",
            "開始分析",
            d_sim,
        ),
        (
            "03",
            "完整數據表",
            "將分析結果整理成表格，方便查看每一組價格、票數與期望值。",
            "查看數據",
            g_simu,
        ),
    ]

    for idx, (num, title, desc, button_text, command) in enumerate(feature_data):
        card = make_card(cards)
        card.grid(row=0, column=idx, sticky="nsew", padx=(0 if idx == 0 else 7, 0 if idx == 2 else 7))

        tk.Label(
            card,
            text=num,
            font=("Segoe UI", 11, "bold"),
            bg="#EEF0FF",
            fg=PRIMARY,
            padx=10,
            pady=5,
        ).pack(anchor="w", padx=20, pady=(20, 14))

        tk.Label(
            card,
            text=title,
            font=("Microsoft JhengHei UI", 14, "bold"),
            bg=CARD,
            fg=TEXT,
        ).pack(anchor="w", padx=20)

        tk.Label(
            card,
            text=desc,
            font=("Microsoft JhengHei UI", 9),
            bg=CARD,
            fg=MUTED,
            justify="left",
            wraplength=260,
        ).pack(anchor="w", padx=20, pady=(8, 18))

        ttk.Button(
            card,
            text=button_text,
            style="Secondary.TButton",
            command=command,
        ).pack(anchor="w", padx=20, pady=(0, 20))

    footer = tk.Frame(container, bg=BG)
    footer.pack(fill="x", pady=(18, 0))

    tk.Label(
        footer,
        text="提示：第一次使用請先從「票數與偏好設定」開始。",
        font=("Microsoft JhengHei UI", 9),
        bg=BG,
        fg=MUTED,
    ).pack(side="left")

    ttk.Button(
        footer,
        text="結束程式",
        style="Danger.TButton",
        command=win.destroy,
    ).pack(side="right")


# =========================
# 程式入口
# =========================
win = tk.Tk()
win.title("一番賞期望值系統")
win.configure(bg=BG)
center_window(win, 1120, 760)
win.minsize(980, 700)
setup_styles()

# 保留上方選單，也新增更好用的主畫面按鈕
menu = tk.Menu(win)
win.config(menu=menu)

simu_menu = tk.Menu(menu, tearoff=False)
menu.add_cascade(label="一番賞購買", menu=simu_menu)
simu_menu.add_command(label="票數設定", command=dy_simu)
simu_menu.add_separator()
simu_menu.add_command(label="結束", command=win.destroy)

vision = tk.Menu(menu, tearoff=False)
menu.add_cascade(label="視覺化", menu=vision)
vision.add_command(label="3D 預測期望值視覺化", command=d_sim)
vision.add_command(label="獎品數量長條圖", command=s_n1)

view_menu = tk.Menu(menu, tearoff=False)
menu.add_cascade(label="數據", menu=view_menu)
view_menu.add_command(label="一番賞模擬與期望值數據", command=g_simu)

build_home()
win.mainloop()

In [1]:
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib.figure import Figure
import tkinter as tk
from tkinter import messagebox as msgbox
from tkinter import ttk
import sqlite3 as s
import csv
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import f_regression
from sklearn import metrics
from tkinter import font, PhotoImage
import random as r
import numpy as np
from PIL import ImageTk, Image
import os
plt.rcParams['font.family'] = ['PingFang TC', 'AppleGothic', 'Heiti TC']
plt.rcParams['axes.unicode_minus'] = False
result = []
vas = []
much = []
# 全域變數初始化
count = {}
total_tickets = 0
price_1 = 0
price_2 = 0 
price_3 = 0
p1 = 0
p2 = 0
p3 = 0
def g_data():
    # 檢查必要的變數是否存在
    try:
        if 'result' not in globals() or 'vas' not in globals() or 'much' not in globals():
            msgbox.showerror("錯誤", "請先執行「預測購買張數的期望值視覺化」以生成數據！")
            return
        
        if len(result) == 0 or len(vas) == 0 or len(much) == 0:
            msgbox.showerror("錯誤", "沒有可顯示的數據，請先執行視覺化分析！")
            return
            
    except NameError:
        msgbox.showerror("錯誤", "請先執行「預測購買張數的期望值視覺化」以生成數據！")
        return
    
    g_sat = []
    for i in range(len(much)):
        g_sat.append([result[i], vas[i], much[i]])
    
    # 清空之前的數據
    for item in tree1.get_children():
        tree1.delete(item)
    
    # 插入數據
    i = 0
    for j in g_sat:  # [價格, 票數, 期望值]
        tree1.insert(parent='', index=i, iid=i, text='', values=(j[0], j[1], j[2]))
        i = i + 1
    
    # 修正總和計算 - 使用正確的變數名稱
    tree1.insert(parent='', index=i, iid=i, text='', values=('總和', sum(result), sum(much)))    

def g_simu():
    global tree1
    w_5=tk.Tk()
    w_5.title('一番賞期望值數據')
    w_5.geometry('600x450')    
    tk.Button(w_5,text ='查看期望值',width='50', command=g_data).place(x=60,y=10)
    tree1=ttk.Treeview(w_5,show='headings',height=20,column=('價格','票數','購買理論期望值'))#顯示層級數據
    tree1.column('價格', anchor='center', width=100)#對齊方式，可選n, ne, e, se, s, sw, w, nw, center
    tree1.column('票數', anchor='center', width=100)
    tree1.column('購買理論期望值', anchor='center', width=100)
    tree1.heading('價格', text='價格')
    tree1.heading('票數', text='票數')
    tree1.heading('購買理論期望值', text='購買理論期望值')
    tree1.place(x=150,y=50)   
    w_5.mainloop()
    
def d_sim():
    global count, total_tickets, price_1, price_2, price_3,result,vas,much
    
    # 檢查是否已經設定過票數
    if not count or total_tickets == 0:
        msgbox.showerror("錯誤", "請先進行票數設定！")
        return
    
    # 檢查必要的變數是否存在
    try:
        combo1_value = combo1.get()
        combo2_value = combo2.get() 
        combo3_value = combo3.get()
        
        # 取得對應的市場價格
        price_map = {
            combo1.get(): c1.get(),
            combo2.get(): c2.get(),
            combo3.get(): c3.get(),
        }
        
        # 定義 c1_val, c2_val, c3_val 為對應的市場價格
        c1_val = price_map[combo1_value]
        c2_val = price_map[combo2_value]
        c3_val = price_map[combo3_value]
        
    except Exception as e:
        msgbox.showerror("錯誤", "請先完成票數設定！")
        return
    
    like = list(range(100, 501, 50))  # 價格（x）
    ticket_range = list(range(1, 82, 10))  # 張數（y）

    result = []
    vas = []
    much = []

    for price in like:
        for tickets in ticket_range:
            for val, combo in zip([c1_val, c2_val, c3_val], [combo1_value, combo2_value, combo3_value]):
                expect = val * count[combo] / total_tickets * tickets
                cost = price * tickets
                net = round(expect - cost, 2)

                result.append(price)       # x
                vas.append(tickets)        # y
                much.append(net)           # z


    w_3 = tk.Toplevel()
    w_3.title('3D視覺化')
    w_3.geometry('800x700')
    w_3.config(bg='#ADD8E6')
    
    fig = Figure(figsize=(10, 8), dpi=100)
    ax = fig.add_subplot(111, projection='3d')
    
    value = np.random.uniform(0, 100, len(much))
    color_map = plt.cm.get_cmap('prism')
    
    scatter = ax.scatter(result, vas, much, c=value, cmap=color_map, alpha=0.7, s=20)
    
    ax.set_xlabel('價格變化', fontsize=12)
    ax.set_ylabel('購買票數', fontsize=12)
    ax.set_zlabel('期望值', fontsize=12)
    ax.set_title('一番賞預測期望值3D視覺化', fontsize=14)
    
    canvas = FigureCanvasTkAgg(fig, master=w_3)
    canvas.draw()
    canvas.get_tk_widget().pack(side=tk.TOP, fill=tk.BOTH, expand=1)
    
    
def s_n1():
    try:
        ticket = [A.get(), B.get(), C.get(), D.get(), E.get(), F.get(),G.get()]
        plot_win = tk.Toplevel()
        plot_win.title("獎品數量長條圖")
        plot_win.geometry("600x400")
        fig = Figure(figsize=(6, 4), dpi=100)
        ax = fig.add_subplot(111)
        ax.bar(pc, ticket, color='skyblue')
        ax.set_title('獎品數量分佈')
        ax.set_xlabel('獎品種類')
        ax.set_ylabel('張數')
        
        canvas = FigureCanvasTkAgg(fig, master=plot_win)
        canvas.draw()
        canvas.get_tk_widget().pack(fill=tk.BOTH, expand=1)
    except:
        msgbox.showerror("錯誤", "請先進行票數設定！")

def s_n():
    global count, p1, p2, p3, total_tickets, price_1, price_2, price_3
    
    count = {
        "A賞": A.get(),
        "B賞": B.get(),
        "C賞": C.get(),
        "D賞": D.get(),
        "E賞": E.get(),
        "F賞": F.get(),
        "G賞": G.get(),
    }
    total_tickets = sum(count.values())
    
    if total_tickets == 0:
        msgbox.showerror("錯誤", "總票數不能為0！")
        return
    
    price_map = {
        combo1.get(): c1.get(),
        combo2.get(): c2.get(),
        combo3.get(): c3.get(),
    }
    price_1 = price_map[combo1.get()]
    price_2 = price_map[combo2.get()]
    price_3 = price_map[combo3.get()]

    buy_count = int(combobox.get())
    price_per_ticket = m1.get()

    prospect = (price_1 * count[combo1.get()] / total_tickets + 
                price_2 * count[combo2.get()] / total_tickets + 
                price_3 * count[combo3.get()] / total_tickets) * buy_count
    
    p1 = price_1 * count[combo1.get()] / total_tickets
    p2 = price_2 * count[combo2.get()] / total_tickets
    p3 = price_3 * count[combo3.get()] / total_tickets
    
    cost = buy_count * price_per_ticket
    final = prospect - cost

    msgbox.showinfo("期望值", f"期望值是 {final:.2f}")

def dy_simu(): 
    global A, B, C, D, E, F, G, combobox, m1, m2, combo1, combo2, combo3, c1, c2, c3, pc, values
    
    w_1 = tk.Toplevel()
    w_1.title('一番賞期望值模擬')
    w_1.geometry('500x300')
    
    A = tk.IntVar()
    B = tk.IntVar()
    C = tk.IntVar()
    D = tk.IntVar()
    E = tk.IntVar()
    F = tk.IntVar()
    G = tk.IntVar()
    m1 = tk.IntVar()
    c1 = tk.IntVar()
    c2 = tk.IntVar()
    c3 = tk.IntVar()
    
    tk.Label(w_1, text='A賞張數', font='14').place(x=10, y=10)
    tk.Entry(w_1, width=10, textvariable=A).place(x=10, y=40)
    tk.Label(w_1, text='B賞張數', font='14').place(x=80, y=10)
    tk.Entry(w_1, width=10, textvariable=B).place(x=80, y=40)
    tk.Label(w_1, text='C賞張數', font='14').place(x=150, y=10)
    tk.Entry(w_1, width=10, textvariable=C).place(x=150, y=40)
    tk.Label(w_1, text='D賞張數', font='14').place(x=210, y=10)
    tk.Entry(w_1, width=10, textvariable=D).place(x=210, y=40)
    tk.Label(w_1, text='E賞張數', font='14').place(x=280, y=10)
    tk.Entry(w_1, width=10, textvariable=E).place(x=280, y=40)
    tk.Label(w_1, text='F賞張數', font='14').place(x=350, y=10)
    tk.Entry(w_1, width=10, textvariable=F).place(x=350, y=40)
    tk.Label(w_1, text='G賞張數', font='14').place(x=420, y=10)
    tk.Entry(w_1, width=10, textvariable=G).place(x=420, y=40)
    
    values = list(range(1, 81))
    combobox = ttk.Combobox(w_1, values=values)
    tk.Label(w_1, text='預算購買張數', font='14').place(x=80, y=83)
    combobox.place(x=170, y=82)
    combobox.set(1)
    
    tk.Label(w_1, text='一張的價格', font='14').place(x=80, y=120)
    tk.Entry(w_1, width=10, textvariable=m1).place(x=170, y=120)
    
    tk.Label(w_1, text='你第一想要的獎品', font='14').place(x=60, y=150)
    pc = ["A賞", "B賞", "C賞", "D賞", "E賞", "F賞","G賞"]
    combo1 = ttk.Combobox(w_1, values=pc)
    combo1.place(x=170, y=150)
    combo1.set("A賞")
    
    tk.Label(w_1, text='你第二想要的獎品', font='14').place(x=60, y=180)
    combo2 = ttk.Combobox(w_1, values=pc)
    combo2.place(x=170, y=180)
    combo2.set("B賞")
    
    tk.Label(w_1, text='你第三想要的獎品', font='14').place(x=60, y=210)
    combo3 = ttk.Combobox(w_1, values=pc)
    combo3.place(x=170, y=210)
    combo3.set("C賞")
    
    tk.Entry(w_1, width=10, textvariable=c1).place(x=380, y=150)
    tk.Entry(w_1, width=10, textvariable=c2).place(x=380, y=180)
    tk.Entry(w_1, width=10, textvariable=c3).place(x=380, y=210)
    tk.Label(w_1, text='市場價格', font='14').place(x=380, y=130)
    
    tk.Button(w_1, text='計算期望值', width='20', command=s_n).place(x=10, y=260)
    tk.Button(w_1, text="長條圖", width='20', command=s_n1).place(x=280, y=260)
def load_and_display_image():
    """載入桌面圖片並顯示"""
    try:
        desktop_path = os.path.join(os.path.expanduser("~"), "Desktop")
        img_path = os.path.join(desktop_path, "一番賞.png")
        
        if os.path.exists(img_path):
            # 載入圖片
            img = Image.open(img_path)
            
            # 調整圖片大小以適應視窗
            img = img.resize((400, 200), Image.Resampling.LANCZOS)
            
            # 轉換為 tkinter 圖片物件
            tk_img = ImageTk.PhotoImage(img, master=win)
            
            # 在視窗中顯示圖片
            img_label = tk.Label(win, image=tk_img, bg='#FA8072')
            img_label.place(x=100, y=10)
            img_label.image = tk_img  # 保持圖片引用，避免被垃圾回收
            
            return img_label, tk_img
        else:
            # 如果找不到圖片，創建一個佔位符
            placeholder_label = tk.Label(
                win, 
                text="一番賞圖片\n(桌面上找不到『一番賞.png』檔案)", 
                font=("Arial", 14), 
                bg='#FFC9C9',
                width=50,
                height=10
            )
            placeholder_label.place(x=100, y=30)
            msgbox.showwarning("警告", "桌面上找不到『一番賞.png』檔案")
            return placeholder_label, None
    except Exception as e:
        # 處理任何載入圖片時的錯誤
        error_label = tk.Label(
            win, 
            text=f"載入圖片時發生錯誤:\n{str(e)}", 
            font=("Arial", 12), 
            bg='#FFC9C9',
            width=50,
            height=10
        )
        error_label.place(x=150, y=30)
        msgbox.showerror("錯誤", f"載入圖片時發生錯誤: {str(e)}")
        return error_label, None



win = tk.Tk()
win.title("一番賞期望值系統")
win.geometry('600x350')
win.config(bg='#FA8072')
img_label, tk_img = load_and_display_image()
text_label = tk.Label(
    win,
    text="歡迎使用一番賞期望值計算系統！\n請從上方選單中選擇功能。",
    font=("Arial", 30),
    bg="#FFF0F5",
    fg="black",
    justify="center"
)
text_label.place(x=80, y=250)
menu = tk.Menu(win)
win.config(menu=menu)

simu_menu = tk.Menu(menu, tearoff=False)
menu.add_cascade(label="一番賞購買", menu=simu_menu)
simu_menu.add_command(label="票數設定", command=dy_simu)
simu_menu.add_command(label="結束", command=win.destroy)

vision = tk.Menu(menu, tearoff=False)
menu.add_cascade(label="視覺化", menu=vision)
vision.add_command(label="預測購買張數的期望值視覺化", command=d_sim)
view_menu = tk.Menu(menu, tearoff=False)
menu.add_cascade(label="一番賞模擬與期望值數據", menu=view_menu)
view_menu.add_command(label="一番賞模擬與期望值數據", command=g_simu)


win.mainloop()